In [ ]:
import os, sys, subprocess
!lscpu | grep "Model name\|CPU(s):"
!free -h

# Hanya install open_clip_torch tanpa mengutak-atik versi transformers bawaan Kaggle
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "open_clip_torch"])

In [ ]:
# add dependencies ini untuk model varian , uncomment untuk menjalankan
# !pip install -q --upgrade open_clip_torch transformers sentence-transformers sentencepiece protobuf

In [ ]:
import time
start_time = time.time()

In [ ]:
import pandas as pd, torch
import matplotlib.pyplot as plt, textwrap
import numpy as np
from PIL import Image
from IPython.display import display, Image as IPyImage

# Uncomment if use online drive (collab)
#from google.colab import drive

# modul tambahan, untuk mengacak data secara konsisten dan  membersihkan memori RAM secara manual
import gc, random
import torch.nn.functional as F

# library model encoder M1 & M2
import open_clip
from sentence_transformers import SentenceTransformer
from transformers import AutoModel, AutoProcessor

# modul untuk membuka & membaca file zip, json
import zipfile
import json

# untuk mengubah data mentah gambar (bytes) yang dibaca langsung dari dalam .zip menjadi format yang bisa dibuka oleh Python/PIL
from io import BytesIO

# SEED kunci angka acak(identik untuk semua eksperimen)
SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"Environment siap. PyTorch version: {torch.__version__} | Device: CPU | SEED: {SEED}")

In [ ]:
# Uncomment these if use online drive (collab)
# Note: Sesuaiin path_train, path_val, path_images di bawah ini sesuai dengan lokasi file di drive kalian

#drive.mount('/content/drive')
#PATH_TRAIN = "/content/drive/MyDrive/"
#PATH_VAL = "/content/drive/MyDrive/"
#PATH_IMAGES = "/content/drive/MyDrive/"
#PATH_OUTPUT = "/content/drive/MyDrive/....../hasil_evaluasi.json"

In [ ]:
# Uncomment these if use kaggle
# Note: Sesuaiin path_train, path_val, path_images di bawah ini sesuai dengan lokasi file di datasets kalian
PATH_TRAIN = "/kaggle/input/datasets/stinart/cleaned-sea-vl-indonesian-only/"
PATH_VAL = "/kaggle/input/datasets/stinart/cleaned-sea-vl-indonesian-only/"
PATH_IMAGES = "/kaggle/input/datasets/stinart/cleaned-sea-vl-indonesian-only/"

PATH_OUTPUT = "/kaggle/working/hasil_evaluasi.json"

# Fungsi untuk menyimpan hasil vektor embbedding, agar tidak menghitung ulang
PATH_CACHE  = "/kaggle/working/cache_m2/"
os.makedirs(PATH_CACHE, exist_ok=True)

In [ ]:
# Uncomment these if use local
#PATH_TRAIN = "../data/"
#PATH_VAL = "../data/"
#PATH_IMAGES = "../data/"
#PATH_OUTPUT = "../data/Output/hasil_evaluasi.json"

In [ ]:
# Read train_clean.csv
# df = pd.read_csv("train_clean.csv")

# Read val_clean.csv
df = pd.read_csv(PATH_VAL + "val_clean.csv")
df_new = df #print(df_new["file"].head)
df_new["file"] = [(PATH_IMAGES + f) for f in df_new["file"]] #print(df_new["file"].head)
imgs  = [Image.open(f).convert("RGB") for f in df_new["file"]]
cap_id = list(df["caption_id"])
cap_en = list(df["caption_en"])

# Keluaran untuk memastikan dataset
print(f"Berhasil memuat {len(imgs)} gambar, {len(cap_id)} caption ID, dan {len(cap_en)} caption EN.")

# Inisialiasi Varian Model 

Lanjut kita mengubah gambar dan teks (Indonesia & Inggris) menjadi vektor angka (embedding) menggunakan 4 model berbeda, sekaligus mengelola penyimpanan cache dan memori RAM.

In [ ]:
# Ekstraksi Model Google SigLIP 2
def encode_siglip2_hf(model_id, imgs, cap_id, cap_en, img_bs=32, txt_bs=64): # Pemrosesan Per Batch
    """Ekstraksi untuk Varian 2: google/siglip2-base-patch16-224 via transformers"""
    processor = AutoProcessor.from_pretrained(model_id)
    model = AutoModel.from_pretrained(model_id)
    model.eval()
    model.requires_grad_(False) # Untuk Memastikan seluruh bobot pretrained tetap frozen
        

    img_embs, ti_embs, te_embs = [], [], []

    with torch.no_grad():
        # Ekstraksi embedding gambar
        for i in range(0, len(imgs), img_bs):
            inputs = processor(
                images=imgs[i : i + img_bs],
                return_tensors="pt"
            )

            outputs = model.get_image_features(**inputs)
            feats = outputs.pooler_output
            feats = F.normalize(feats, p=2, dim=-1)

            img_embs.append(feats.cpu().numpy())

        # Ekstraksi embedding caption Indonesia
        for i in range(0, len(cap_id), txt_bs):
            batch_txt = [str(t) for t in cap_id[i : i + txt_bs]]

            inputs = processor(
                text=batch_txt,
                padding="max_length",
                truncation=True,
                return_tensors="pt"
            )

            outputs = model.get_text_features(**inputs)
            feats = outputs.pooler_output
            feats = F.normalize(feats, p=2, dim=-1)

            ti_embs.append(feats.cpu().numpy())

        # Ekstraksi embedding caption Inggris
        for i in range(0, len(cap_en), txt_bs):
            batch_txt = [str(t) for t in cap_en[i : i + txt_bs]]

            inputs = processor(
                text=batch_txt,
                padding="max_length",
                truncation=True,
                return_tensors="pt"
            )

            outputs = model.get_text_features(**inputs)
            feats = outputs.pooler_output
            feats = F.normalize(feats, p=2, dim=-1)

            te_embs.append(feats.cpu().numpy())

    del model, processor
    gc.collect()

    return np.vstack(img_embs), np.vstack(ti_embs), np.vstack(te_embs)


# Ekstraksi untuk Model Berbasis OpenCLIP
def encode_openclip_model(arch_name, pretrained_name, imgs, cap_id, cap_en, img_bs=32, txt_bs=64, lang_id=None, lang_en=None):
    """
    Ekstraksi untuk:
    - Varian 1: laion/CLIP-ViT-B-32-xlm-roberta-base-laion5B-s13B-b90k
    - Varian 3: visheratin/nllb-clip-base-siglip (dengan kode bahasa NLLB-200: ind_Latn & eng_Latn)
    """
    model, _, preprocess = open_clip.create_model_and_transforms(arch_name, pretrained=pretrained_name, device="cpu")
    tokenizer = open_clip.get_tokenizer(arch_name)
    model.eval()
    model.requires_grad_(False)  # Untuk Memastikan seluruh bobot pretrained tetap frozen
    
    img_embs, ti_embs, te_embs = [], [], []
    with torch.no_grad():
        for i in range(0, len(imgs), img_bs):
            batch_tensors = torch.stack([preprocess(im) for im in imgs[i : i + img_bs]])
            feats = F.normalize(model.encode_image(batch_tensors), p=2, dim=-1)
            img_embs.append(feats.cpu().numpy())

        for i in range(0, len(cap_id), txt_bs):
            batch_txt = [str(t) for t in cap_id[i : i + txt_bs]]
            
            if lang_id:
                tokenizer.set_language(lang_id)
            tokens = tokenizer(batch_txt) # memberitahu tokenizer bahasa mana yang sedang dibaca
            
            feats = F.normalize(model.encode_text(tokens), p=2, dim=-1)
            ti_embs.append(feats.cpu().numpy())

        for i in range(0, len(cap_en), txt_bs):
            batch_txt = [str(t) for t in cap_en[i : i + txt_bs]]

            if lang_en:
                tokenizer.set_language(lang_en)
            tokens = tokenizer(batch_txt) # memberitahu tokenizer bahasa mana yang sedang dibaca

            feats = F.normalize(model.encode_text(tokens), p=2, dim=-1)
            te_embs.append(feats.cpu().numpy())

    del model, preprocess, tokenizer
    gc.collect()
    return np.vstack(img_embs), np.vstack(ti_embs), np.vstack(te_embs)

# Manajer Utama & Pengatur Cache
def run_encoding(variant_key, imgs, cap_id, cap_en, use_cache=True):
    """Menjalankan ekstraksi sesuai 4 model rencana kita + simpan cache"""
    cache_file = os.path.join(PATH_CACHE, f"{variant_key}.pt")
    t0 = time.time()

    if use_cache and os.path.exists(cache_file):
        data = torch.load(cache_file, map_location="cpu", weights_only=False)
        print(f"[{variant_key}] Loaded dari cache ({data['elapsed_sec']} dtk)")
        return data["E"], data["Ti"], data["Te"], data["elapsed_sec"]

    print(f"[{variant_key}] Mengekstrak embedding di CPU...")

    if variant_key == "baseline_m1":
        # 1. Baseline M1 (100% kode asli Peran 2)
        img_model = SentenceTransformer("sentence-transformers/clip-ViT-B-32")
        txt_model = SentenceTransformer("sentence-transformers/clip-ViT-B-32-multilingual-v1")
        E  = img_model.encode(imgs,   batch_size=32, normalize_embeddings=True)
        Ti = txt_model.encode(cap_id, batch_size=64, normalize_embeddings=True)
        Te = img_model.encode(cap_en, batch_size=64, normalize_embeddings=True)
        del img_model, txt_model
        gc.collect()

    elif variant_key == "varian1_laion5b":
        # 2. Varian 1: LAION-5B XLM-RoBERTa-Base ViT-B-32
        E, Ti, Te = encode_openclip_model(
            arch_name="xlm-roberta-base-ViT-B-32",
            pretrained_name="laion5b_s13b_b90k",
            imgs=imgs, cap_id=cap_id, cap_en=cap_en
        )

    elif variant_key == "varian2_siglip2":
        # 3. Varian 2: Google SigLIP 2 Base Patch16-224
        E, Ti, Te = encode_siglip2_hf(
            model_id="google/siglip2-base-patch16-224",
            imgs=imgs, cap_id=cap_id, cap_en=cap_en
        )

    elif variant_key == "varian3_nllb_clip":
        # 4. Varian 3: NLLB-CLIP Base SigLIP (visheratin/nllb-clip-base-siglip)
        E, Ti, Te = encode_openclip_model(
            arch_name="nllb-clip-base-siglip",
            pretrained_name="v1",
            imgs=imgs, cap_id=cap_id, cap_en=cap_en,
            lang_id="ind_Latn", lang_en="eng_Latn"
        )

    else:
        raise ValueError(f"Varian {variant_key} tidak dikenal.")

    elapsed_sec = round(time.time() - t0, 2) # Hitung Waktu & Simpan Cache
    torch.save({"E": E, "Ti": Ti, "Te": Te, "elapsed_sec": elapsed_sec}, cache_file)
    print(f"[{variant_key}] Selesai dalam {elapsed_sec} dtk | Shape: E={E.shape}, Ti={Ti.shape}, Te={Te.shape}")
    return E, Ti, Te, elapsed_sec

print("Fungsi ekstraksi ke-4 model siap!")

# Smoke test 

Lanjut kita mencoba model dengan 2 data saja sebelum memproses ribuan data asli.tujuannya untuk memastikan fungsi dasar suatu program atau model berjalan tanpa error fatal sebelum menjalankan proses penuh.

In [ ]:
# Smoke test untuk memastikan keempat encoder dapat menghasilkan embedding yang valid.
# Pengujian memakai dua sampel dan tidak menggunakan atau membuat cache eksperimen.

N_SMOKE = 2 # Mengambil 2 gambar dan 2 pasangan caption (ID & EN) pertama sebagai bahan uji coba.
smoke_imgs = imgs[:N_SMOKE]
smoke_cap_id = cap_id[:N_SMOKE]
smoke_cap_en = cap_en[:N_SMOKE]

assert len(smoke_imgs) == N_SMOKE
assert len(smoke_cap_id) == N_SMOKE
assert len(smoke_cap_en) == N_SMOKE

# cek hasil smoke, mencetak status LOLOS kalau sesuai requirement
def check_smoke_result(variant_key, E_test, Ti_test, Te_test, elapsed):
    """Memeriksa jumlah data, dimensi, nilai numerik, dan normalisasi embedding."""
    embeddings = {
        "Image": E_test,
        "Caption ID": Ti_test,
        "Caption EN": Te_test
    }

    assert E_test.shape[0] == N_SMOKE
    assert Ti_test.shape[0] == N_SMOKE
    assert Te_test.shape[0] == N_SMOKE
    assert E_test.shape[1] == Ti_test.shape[1] == Te_test.shape[1]

    for name, values in embeddings.items():
        assert np.isfinite(values).all(), (
            f"{variant_key}: ditemukan NaN/Inf pada {name}."
        )
        norms = np.linalg.norm(values, axis=1)
        assert np.allclose(norms, 1.0, atol=1e-3), (
            f"{variant_key}: embedding {name} belum ternormalisasi."
        )

    print(
        f"{variant_key:20s} | "
        f"dimensi={E_test.shape[1]:4d} | "
        f"waktu={elapsed:8.2f} detik | LOLOS"
    )

# Menguji keempat varian satu per satu tanpa menyimpan file cache
# Setelah setiap model diuji, variabelnya langsung dihapus (del ... dan gc.collect()) agar RAM tidak penuh.
def smoke_test_all_variants():
    """Menjalankan smoke test satu per satu agar penggunaan memori tetap terkendali."""
    results = []

    print("Smoke test menggunakan 2 gambar dan 2 caption per bahasa.")
    print("-" * 78)

    smoke_variants = [
        "baseline_m1", "varian1_laion5b",
        "varian2_siglip2", "varian3_nllb_clip"
    ]
    for variant_key in smoke_variants:
        t0 = time.time()

        if variant_key == "baseline_m1":
            image_encoder = SentenceTransformer(
                "sentence-transformers/clip-ViT-B-32"
            )
            text_encoder = SentenceTransformer(
                "sentence-transformers/clip-ViT-B-32-multilingual-v1"
            )
            image_encoder.eval()
            text_encoder.eval()

            with torch.no_grad():
                E_test = image_encoder.encode(
                    smoke_imgs, batch_size=2,
                    normalize_embeddings=True, convert_to_numpy=True,
                    show_progress_bar=False
                )
                Ti_test = text_encoder.encode(
                    smoke_cap_id, batch_size=2,
                    normalize_embeddings=True, convert_to_numpy=True,
                    show_progress_bar=False
                )
                Te_test = image_encoder.encode(
                    smoke_cap_en, batch_size=2,
                    normalize_embeddings=True, convert_to_numpy=True,
                    show_progress_bar=False
                )

            del image_encoder, text_encoder
            gc.collect()

        elif variant_key == "varian1_laion5b":
            E_test, Ti_test, Te_test = encode_openclip_model(
                arch_name="xlm-roberta-base-ViT-B-32",
                pretrained_name="laion5b_s13b_b90k",
                imgs=smoke_imgs,
                cap_id=smoke_cap_id,
                cap_en=smoke_cap_en,
                img_bs=2,
                txt_bs=2
            )

        elif variant_key == "varian2_siglip2":
            E_test, Ti_test, Te_test = encode_siglip2_hf(
                model_id="google/siglip2-base-patch16-224",
                imgs=smoke_imgs,
                cap_id=smoke_cap_id,
                cap_en=smoke_cap_en,
                img_bs=2,
                txt_bs=2
            )

        elif variant_key == "varian3_nllb_clip":
            E_test, Ti_test, Te_test = encode_openclip_model(
                arch_name="nllb-clip-base-siglip",
                pretrained_name="v1",
                imgs=smoke_imgs,
                cap_id=smoke_cap_id,
                cap_en=smoke_cap_en,
                img_bs=2,
                txt_bs=2,
                lang_id="ind_Latn",
                lang_en="eng_Latn"
            )

        elapsed = time.time() - t0
        check_smoke_result(
            variant_key, E_test, Ti_test, Te_test, elapsed
        )

        results.append({
            "varian": variant_key,
            "dimensi_embedding": E_test.shape[1],
            "waktu_smoke_test_detik": round(elapsed, 2),
            "status": "LOLOS"
        })

        del E_test, Ti_test, Te_test
        gc.collect()

    return pd.DataFrame(results)


smoke_test_results = smoke_test_all_variants()
display(smoke_test_results)


# Ekstraksi embedding penuh

Setelah seluruh smoke test berstatus LOLOS, kita menjalankan ekstraksi untuk seluruh data validation pada setiap varian. Cache lama dilewati pada proses pertama agar embedding benar-benar dihitung ulang.

In [ ]:
# Ekstraksi embedding untuk seluruh varian pada data validation.
# use_cache=False memastikan hasil dihitung ulang, bukan diambil dari cache sebelumnya.

embeddings_dict = {}
ringkasan_ekstraksi = []
jumlah_data = len(imgs)

assert jumlah_data == len(cap_id) == len(cap_en), (
    "Jumlah gambar dan caption tidak sama. Periksa kembali data validation."
) # Memastikan jumlah gambar dan caption id en sama

for variant_key in DAFTAR_VARIAN:
    print(f"\nMemproses {variant_key} ({jumlah_data} sampel)...")

    E_v, Ti_v, Te_v, waktu_v = run_encoding(
        variant_key,
        imgs,
        cap_id,
        cap_en,
        use_cache=False # memaksa program menghitung ulang embedding dari nol
    )

    # Memastikan setiap embedding tetap sejajar dengan baris dataset.
    assert E_v.shape[0] == jumlah_data
    assert Ti_v.shape[0] == jumlah_data
    assert Te_v.shape[0] == jumlah_data
    assert E_v.shape[1] == Ti_v.shape[1] == Te_v.shape[1]

    # Memeriksa nilai embedding sebelum diserahkan ke tahap evaluasi.
    for nama, arr in [("Image", E_v), ("Caption ID", Ti_v), ("Caption EN", Te_v)]:
        assert np.isfinite(arr).all(), (
            f"{variant_key}: ditemukan NaN/Inf pada {nama}."
        )
        norma = np.linalg.norm(arr, axis=1)
        assert np.allclose(norma, 1.0, atol=1e-3), (
            f"{variant_key}: embedding {nama} belum ternormalisasi."
        )

    embeddings_dict[variant_key] = {
        "E": E_v,
        "Ti": Ti_v,
        "Te": Te_v,
        "waktu_jalan_detik": waktu_v
    }

    ringkasan_ekstraksi.append({
        "varian": variant_key,
        "jumlah_data": jumlah_data,
        "dimensi_embedding": int(E_v.shape[1]),
        "waktu_ekstraksi_detik": waktu_v,
        "status": "BERHASIL"
    })


ringkasan_ekstraksi = pd.DataFrame(ringkasan_ekstraksi)
display(ringkasan_ekstraksi)

In [ ]:
# Menyimpan embedding dan metadata agar dapat digunakan oleh Peran 4 yaitu evaluator dan uji sulit 
import shutil
from pathlib import Path

EXPORT_DIR = "/kaggle/working/M2_Experiment_Results"
os.makedirs(EXPORT_DIR, exist_ok=True) 

image_ids = df_new["image_id"].astype(str).tolist()
assert len(image_ids) == len(imgs) # Mengambil daftar image_id secara berurutan dan memastikan jumlahnya sama persis dengan jumlah gambar yang diekstrak 

MODEL_INFO = {
    "baseline_m1": {
        "checkpoint": "sentence-transformers/clip-ViT-B-32 + sentence-transformers/clip-ViT-B-32-multilingual-v1",
        "backend": "SentenceTransformers"
    },
    "varian1_laion5b": {
        "checkpoint": "laion/CLIP-ViT-B-32-xlm-roberta-base-laion5B-s13B-b90k",
        "backend": "OpenCLIP"
    },
    "varian2_siglip2": {
        "checkpoint": "google/siglip2-base-patch16-224",
        "backend": "Transformers"
    },
    "varian3_nllb_clip": {
        "checkpoint": "visheratin/nllb-clip-base-siglip",
        "backend": "OpenCLIP",
        "text_language_codes": {"caption_id": "ind_Latn", "caption_en": "eng_Latn"}
    }
}

ringkasan_file = []

for variant_key in DAFTAR_VARIAN:
    data = embeddings_dict[variant_key]
    folder_varian = os.path.join(EXPORT_DIR, variant_key)
    os.makedirs(folder_varian, exist_ok=True)

    # Menyimpan tiga jenis embedding sebagai file terpisah.
    np.save(os.path.join(folder_varian, "E.npy"), np.asarray(data["E"], dtype=np.float32))
    np.save(os.path.join(folder_varian, "Ti.npy"), np.asarray(data["Ti"], dtype=np.float32))
    np.save(os.path.join(folder_varian, "Te.npy"), np.asarray(data["Te"], dtype=np.float32))

    # inisialisasi metadata.json
    # berisi nama checkpoint model yang dipakai, dimensi vektor, waktu proses, nilai SEED, image_ids_in_dataset_order
    metadata_varian = {
        "variant": variant_key,
        "model": MODEL_INFO[variant_key],
        "dataset_file": "val_clean.csv",
        "split": "validation",
        "num_samples": len(image_ids),
        "embedding_dimension": int(data["E"].shape[1]),
        "embedding_files": {
            "E": "E.npy (image embeddings)",
            "Ti": "Ti.npy (Indonesian caption embeddings)",
            "Te": "Te.npy (English caption embeddings)"
        },
        "normalization": "L2-normalized",
        "pretrained_weights_frozen": True,
        "elapsed_seconds_including_model_loading": float(data["waktu_jalan_detik"]),
        "seed": SEED,
        "row_alignment": "Baris ke-i pada E, Ti, dan Te mengacu pada image_id ke-i di bawah ini.",
        "image_ids_in_dataset_order": image_ids
    }

    with open(os.path.join(folder_varian, "metadata.json"), "w", encoding="utf-8") as f:
        json.dump(metadata_varian, f, indent=2, ensure_ascii=False)

    ringkasan_file.append({
        "varian": variant_key,
        "checkpoint": MODEL_INFO[variant_key]["checkpoint"],
        "jumlah_data": len(image_ids),
        "dimensi_embedding": int(data["E"].shape[1]),
        "waktu_ekstraksi_detik": data["waktu_jalan_detik"],
        "folder_output": variant_key
    })

summary_df = pd.DataFrame(ringkasan_file) # berisi nama model, dimensi, dan kecepatan ekstraksi
summary_df.to_csv(os.path.join(EXPORT_DIR, "experiment_summary.csv"), index=False)

# Membuat satu arsip agar seluruh hasil mudah diunduh dari Kaggle.
ZIP_PATH = shutil.make_archive(EXPORT_DIR, "zip", root_dir=EXPORT_DIR)

print("Semua embedding dan metadata berhasil disimpan.")
print(f"Folder hasil : {EXPORT_DIR}")
print(f"Arsip ZIP    : {ZIP_PATH}")
display(summary_df)

In [ ]:
# mengecek lagi hasil embedding yg di compress
ZIP_PATH = "/kaggle/working/M2_Experiment_Results.zip"

dimensi_model = {
    "baseline_m1": 512,
    "varian1_laion5b": 512,
    "varian2_siglip2": 768,
    "varian3_nllb_clip": 768
}

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    assert z.testzip() is None, "Arsip ZIP rusak."

    nama_file = set(z.namelist())
    assert "experiment_summary.csv" in nama_file

    for variant, dim in dimensi_model.items():
        print(f"\nMemeriksa {variant}...")

        # Memastikan seluruh file tersedia.
        files = ["E.npy", "Ti.npy", "Te.npy", "metadata.json"]
        assert all(f"{variant}/{f}" in nama_file for f in files)

        # Memeriksa metadata.
        metadata = json.loads(z.read(f"{variant}/metadata.json"))
        assert metadata["num_samples"] == 275
        assert metadata["embedding_dimension"] == dim
        assert len(metadata["image_ids_in_dataset_order"]) == 275

        # Memeriksa embedding.
        for file in ["E.npy", "Ti.npy", "Te.npy"]:
            arr = np.load(BytesIO(z.read(f"{variant}/{file}")))

            assert arr.shape == (275, dim)
            assert np.isfinite(arr).all()
            assert np.allclose(
                np.linalg.norm(arr, axis=1), 1.0, atol=1e-3
            )

            print(f"{file}: {arr.shape} | VALID")

        print("Metadata: VALID")

    # Memeriksa ringkasan eksperimen.
    summary = pd.read_csv(BytesIO(z.read("experiment_summary.csv")))
    assert len(summary) == len(dimensi_model)

    print("\nSeluruh arsip dan embedding berhasil diverifikasi.")
    display(summary)

In [ ]:
# Kesamaan Kosinus
sim_img_capId = E @ Ti.T
sim_img_capEn = E @ Te.T
sim_capEn_capId = Te @ Ti.T

print("Kesamaan Kosinus image dengan caption indonesia:\n", sim_img_capId, '\n')
print("Kesamaan Kosinus image dengan caption english:\n", sim_img_capEn, '\n')
print("Kesamaan Kosinus caption english dengan caption indonesia:\n", sim_capEn_capId, '\n')

In [ ]:
print(sim_img_capId.shape)
print(sim_img_capEn.shape)
print(sim_capEn_capId.shape)

In [ ]:
# Metrik Recall@K, K = {1,5,10}
def recall_at_K(A, B, ks=(1, 5, 10)):
    rank = (A @ B.T).argsort(dim=1, descending=True)
    truth = torch.arange(len(A)).unsqueeze(1)
    return {k: float((rank[:, :k] == truth).any(1).float().mean()) for k in ks}

# Metrik mAP@K, K = {1,5,10}
def map_at_K(A, B, ks=(1, 5, 10)):
  rank = (A @ B.T).argsort(dim=1, descending=True)
  truth = torch.arange(len(A), device=A.device).unsqueeze(1)
  results = {}
  for k in ks:
    hits = rank[:, :k] == truth

    # Karena hanya ada 1 jawaban benar per kueri,
    # AP@k = 1 / (posisi_peringkat + 1) jika ketemu, dan 0 jika tidak ketemu.
    # argmax(dim=1) mengambil indeks posisi pertama di mana True muncul.
    pos = hits.int().argmax(dim=1)
    has_hit = hits.any(dim=1)

    ap = torch.where(has_hit, 1.0 / (pos.float() + 1.0), 0.0)
    results[k] = float(ap.mean())
  return results


In [ ]:
print("Recall@1/5/10:")
print("gambar -> teks-ID :", recall_at_K(torch.tensor(E), torch.tensor(Ti)))
print("teks-ID -> gambar :", recall_at_K(torch.tensor(Ti), torch.tensor(E)))
print("gambar -> teks-EN :", recall_at_K(torch.tensor(E), torch.tensor(Te)))
print("teks-EN -> gambar :", recall_at_K(torch.tensor(Te), torch.tensor(E)), '\n')

print("mAP@1/5/10:")
print("gambar -> teks-ID :", map_at_K(torch.tensor(E), torch.tensor(Ti)))
print("teks-ID -> gambar :", map_at_K(torch.tensor(Ti), torch.tensor(E)))
print("gambar -> teks-EN :", map_at_K(torch.tensor(E), torch.tensor(Te)))
print("teks-EN -> gambar :", map_at_K(torch.tensor(Te), torch.tensor(E)))

In [ ]:
end_time = time.time()
total_detik = end_time - start_time
print(f"Total waktu jalan: {total_detik:.2f} detik ({total_detik/60:.2f} menit)")

In [ ]:
import json

hasil_eksperimen = {
    "waktu_jalan_detik": round(end_time - start_time, 2),
    "evaluasi_metrik": {
        "Recall_at_K": {
            "gambar_ke_teks_ID": recall_at_K(torch.tensor(E), torch.tensor(Ti)),
            "teks_ID_ke_gambar": recall_at_K(torch.tensor(Ti), torch.tensor(E)),
            "gambar_ke_teks_EN": recall_at_K(torch.tensor(E), torch.tensor(Te)),
            "teks_EN_ke_gambar": recall_at_K(torch.tensor(Te), torch.tensor(E))
        },
        "mAP_at_K": {
            "gambar_ke_teks_ID": map_at_K(torch.tensor(E), torch.tensor(Ti)),
            "teks_ID_ke_gambar": map_at_K(torch.tensor(Ti), torch.tensor(E)),
            "gambar_ke_teks_EN": map_at_K(torch.tensor(E), torch.tensor(Te)),
            "teks_EN_ke_gambar": map_at_K(torch.tensor(Te), torch.tensor(E))
        }
    }
}

outputLocation = PATH_OUTPUT
with open(outputLocation, "w", encoding="utf-8") as f:
    json.dump(hasil_eksperimen, f, indent=4, ensure_ascii=False)

print(f"Hasil berhasil disimpan di: {outputLocation}")


# **Inference** 

In [ ]:
queries_Inference = [
    "Rok batik warna hitam dengan motif tanaman",
    "Di dalam kereta api kelas eksekutif menuju Yogyakarta",
    "Candi Prambanan adalah kompleks candi Hindu terbesar di Indonesia yang terletak di Yogyakarta, dibangun pada abad ke-9 untuk menghormati Trimurti, yaitu Brahma, Wisnu, dan Siwa.",
    "Pemandangan luas perkebunan teh di kawasan Pangalengan",
    "Sajian lembut kue Putu Ayu khas Nusantara yang beraroma wangi pandan",
    "Panorama pesisir dan perairan di sekitar Pantai Kesirat, wilayah Gunung Kidul",
    "Nasi padang pakai dendeng batokok dan perkedel kentang",
    "Gambar tersebut merupakan kentang balado. Salah satu masakan khas dari Indonesia, merupakan kentang yang dicampur dengan cabai merah.",
    "Bakmi yang sudah diaduk. Berwarna kuning kecokelatan. Di atas piring putih dengan sumpit hitam.",
    "Sepiring nasi goreng dengan telur dadar"
]

Ti_Inference = txt_model.encode(queries_Inference, batch_size=64, normalize_embeddings=True)

kunci = {
    0: {"svl_142113188.jpg"},
    1: {"svl_147525764.jpg"},
    2: {"svl_144425377.jpg"},
    3: {"svl_147562928.jpg"},
    4: {"svl_142644501.jpg"},
    5: {"svl_140084565.jpg"},
    6: {"svl_140083990.jpg"},
    7: {"svl_140084024.jpg"},
    8: {"svl_140084155.jpg"},
    9: {"svl_140084730.jpg"},
}

def Top3Grid(queries, text_embeddings, image_embeddings, df, imgs, kunci=None, k=3, size=(224, 224)):
    kunci = kunci or {}
    sim = torch.tensor(text_embeddings) @ torch.tensor(image_embeddings).T
    rank = sim.argsort(dim=1, descending=True)
    n = len(queries)
    fig, axes = plt.subplots(n, k + 1, figsize=(3.2 * (k + 1), 3.4 * n), gridspec_kw={"width_ratios": [1.4] + [1] * k})
    hit1 = hit3 = dinilai = 0

    for i, q in enumerate(queries):
        axes[i, 0].axis("off")
        axes[i, 0].text(0, 0.5, f"Q{i+1}\n" + textwrap.fill(q, 32), va="center", fontsize=9)

        gt = kunci.get(i)
        benar_ke = None

        for j, idx in enumerate(rank[i, :k].tolist()):
            ax = axes[i, j + 1]
            ax.imshow(imgs[idx].resize(size))
            ax.set_xticks([]); ax.set_yticks([])
            fname = df.iloc[idx]["file"].split("/")[-1]

            if gt is None:
                label = ""
            elif fname in gt:
                label = "Benar"
                if benar_ke is None: benar_ke = j
            else:
                label = "Salah"

            for s in ax.spines.values():
                s.set_edgecolor("black"); s.set_linewidth(1)
            ax.set_title(f"#{j+1} {fname}\nskor {sim[i, idx]:.3f}\n{label}", fontsize=8, color="black")

        if gt is not None:
            dinilai += 1
            hit1 += (benar_ke == 0)
            hit3 += (benar_ke is not None)

    plt.tight_layout(); plt.show()
    if dinilai:
        print(f"Top-1 benar: {hit1}/{dinilai} | Top-3 ada yang benar: {hit3}/{dinilai}")

Top3Grid(queries_Inference, Ti_Inference, E, df_new, imgs, kunci)